# AgentOps Lab 09 - Optimize the trajectory

This notebook starts with a deliberately inefficient agent. It succeeds, but it wastes model calls, repeats searches, repeats log queries, and reflects after it already has enough evidence.

The optimization target is not "minimize tokens." The stronger target is: find the shortest reliable trajectory to a correct result.


## Notebook-first learning contract

This notebook is the primary lesson for this topic. The Python module is not a separate replacement for the lesson; it is the implementation layer that the notebook explains, runs, breaks, and evaluates. Work through the notebook in this order:

1. read the concept model and architecture boundary;
2. inspect the tool/state/policy contracts;
3. run the deterministic implementation;
4. trigger the deliberate failure case;
5. record evaluation, cost, latency, and safety observations; and
6. answer the architecture question before moving on.


## Deep-dive training guide — Optimize for shortest reliable trajectory

### Concepts to master

- latency/cost/token economics
- unnecessary reflection and repeated tool calls
- shortest reliable path versus cheapest individual call

### Implementation walkthrough

`trajectory_optimization.py` compares a wasteful but successful trajectory with a shorter reliable one and computes a simple efficiency score.

### Deliberate failure case

Remove evidence-gathering steps only to reduce tokens. The cost improves, but recommendation support collapses.

### Learner exercise

Design a caching rule that avoids duplicate incident searches while preserving freshness and traceability.

### What to write down

For each run, capture the chosen architecture, tool trajectory, evidence used, rejected alternatives, stop condition, estimated cost, latency, and one sentence explaining whether the architecture was the least autonomous reliable option.


## Engineering checklist for this notebook

Use this checklist as your mini design review before you call the topic complete.

| Area | Question to answer |
| --- | --- |
| Control boundary | Which decisions are made by deterministic code, and which are delegated to the model? |
| Tools | Are tool inputs typed, narrow, authorized, and auditable? |
| State | What state is carried between steps, and what should never become long-term memory? |
| Failure mode | What is the easiest way this design loops, overacts, or fabricates certainty? |
| Evaluation | Which outcome, trajectory, safety, cost, and latency signals prove the design is working? |
| Architecture choice | Why is this architecture simpler or better than the nearest alternative? |


## Inefficient trajectory

```mermaid
flowchart TD
    A["Plan"] --> B["search incidents"]
    B --> C["think"]
    C --> D["query health"]
    D --> E["think"]
    E --> F["search incidents again"]
    F --> G["retrieve runbook"]
    G --> H["think"]
    H --> I["query logs"]
    I --> J["reflection"]
    J --> K["query logs again"]
    K --> L["answer"]
```

It succeeds, but success alone is a blunt metric.

In [ ]:
from pathlib import Path
import sys

repo_root = next((candidate for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (candidate / "curriculum" / "advanced" / "05-incident-response-capstone" / "agentops_lab").exists()), None)
if repo_root is None:
    raise RuntimeError("Run this notebook from inside the repository checkout.")
sys.path.insert(0, str(repo_root / "curriculum" / "advanced" / "05-incident-response-capstone"))

from agentops_lab.trajectory_optimization import INEFFICIENT, OPTIMIZED, compare_profiles, efficiency_score, optimization_rules


## Compare before and after

The optimized path keeps the evidence needed for correctness and removes redundant thinking and tool calls.

In [ ]:
comparison = compare_profiles()
comparison


## Efficiency score

For this teaching lab, use a simple score:

```python
efficiency_score = success / (latency_seconds + cost_weight + trajectory_length)
```

The exact formula is less important than the habit: compare correct runs by latency, cost, and path length, not vibes.

In [ ]:
print("inefficient:", efficiency_score(INEFFICIENT))
print("optimized:", efficiency_score(OPTIMIZED))


## Optimization rules

These are the rules the learner applies to move from the wasteful trajectory to the shorter reliable one.

In [ ]:
for rule in optimization_rules():
    print("-", rule)


## What improved?

The optimized run keeps success while reducing model calls, tool calls, latency, cost, and trajectory length.

In [ ]:
comparison["improvement"]


## Exercises

- Remove `retrieve_runbook` from the optimized trajectory. Does recommendation support still pass?
- Add a cheaper but less reliable path. How do you decide whether to ship it?
- Add a trajectory cache for repeated `search_incidents` calls.
- Compare cost per successful task before and after optimization across ten tasks.

References: [Anthropic: Building effective agents](https://www.anthropic.com/engineering/building-effective-agents), [Anthropic: demystifying evals for AI agents](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents), and [Building AI Agents: From Loops to Teams](https://www.linkedin.com/pulse/building-ai-agents-from-loops-teams-oneplusi-y3atc/).